In [9]:
import logging
from typing import List, Dict, Optional
from ollama import chat, generate

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

MODEL = "saimon:latest"


def summarize_text(text: str) -> Optional[str]:
    try:
        logging.info("Summarizing input text...")
        prompt = f"Summarize the following text clearly and concisely:\n\n{text}"
        
        response = generate(
            model=MODEL,
            prompt=prompt,
            options={'temperature': 0.0}
        )
        
        summary = response.get('response', '')
        logging.info("Summary generated successfully.")
        return summary
    except Exception as e:
        logging.error("Failed to generate summary: %s", e)
        return None


def summarize_conversation(messages: List[Dict[str, str]]) -> Optional[str]:

    if not messages:
        logging.warning("No conversation history to summarize.")
        return None

    try:
        logging.info("Summarizing full conversation context...")
        summary_messages = messages + [
            {'role': 'user', 'content': 'Summarize our conversation so far into bullet points.'}
        ]

        response = chat(
            model=MODEL,
            messages=summary_messages,
            options={'temperature': 0.0},
            stream=False
        )

        summary = response.get('message', {}).get('content', '')
        logging.info("Conversation summary generated successfully.")
        return summary
    except Exception as e:
        logging.error("Failed to summarize conversation context: %s", e)
        return None


def query_llm() -> None:
    messages = []
    
    while True:
        try:
            user_input = input("\nYou (type '/summarize <text>' or '/summarize-chat'): ")
            
            if user_input.strip().lower() in ['exit', 'quit']:
                logging.info("Ending chat session.")
                break

            if not user_input.strip():
                logging.warning("User input was empty.")
                continue

            if user_input.strip().lower() == '/summarize-chat':
                summary = summarize_conversation(messages)
                if summary:
                    logging.info("Conversation Summary:\n%s", summary)
                continue

            if user_input.startswith('/summarize '):
                target_text = user_input[len('/summarize '):].strip()
                if not target_text:
                    logging.warning("No text provided for summarization.")
                    continue
                
                summary = summarize_text(target_text)
                if summary:
                    logging.info("Text Summary Output:\n%s", summary)
                continue

            logging.info("User Prompt: %s", user_input)

            messages.append({'role': 'user', 'content': user_input})

            stream = chat(
                model=MODEL,
                messages=messages,
                options={'temperature': 0.0},
                stream=True,
            )

            full_response = ""
            for chunk in stream:
                content = chunk.get('message', {}).get('content', '')
                full_response += content

            logging.info("Saimon's Response:\n%s", full_response)
            
            messages.append({'role': 'assistant', 'content': full_response})

        except Exception as e:
            logging.error("Failed to execute chat stream: %s", e)


if __name__ == "__main__":
    query_llm()

2026-09-25 10:24:43,136 - INFO - Ending chat session.
